In [ ]:
import duckdb

# CREATE OR REPLACE VIEW games AS: query under games 
# OR REPLACE: no error if already exists
# SELECT * FROM: reads CSV file
duckdb.sql("""
CREATE OR REPLACE VIEW games AS
SELECT * FROM '../data/games_clean_2025_26.csv'
""")

# team games: one row for each team
# SELECT: home team side
# UNION ALL: bind rows from home and away team sides
# SELECT: away team; match by position of columns
duckdb.sql("""
CREATE OR REPLACE VIEW team_games AS
SELECT GAME_ID, GAME_DATE,
       home_team AS team, away_team AS opponent,
       home_pts  AS pts,  away_pts  AS opp_pts,
       TRUE AS is_home, neutral_site
FROM games
UNION ALL
SELECT GAME_ID, GAME_DATE,
       away_team, home_team,
       away_pts,  home_pts,
       FALSE, neutral_site
FROM games
""")

duckdb.sql("SELECT COUNT(*) AS n_rows FROM team_games").df()

,n_rows
0,2460


In [ ]:
# 1: Which teams score the most points

# COUNT: number of games played by each team
# SELECT team: one row per team
duckdb.sql("""
SELECT team,
       ROUND(AVG(pts), 1) AS ppg,
       COUNT(*)           AS games
FROM team_games
GROUP BY team
ORDER BY ppg DESC
""").df()

,team,ppg,games
0,DEN,122.1,82
1,MIA,120.9,82
2,SAS,119.8,82
3,CLE,119.5,82
4,OKC,119.0,82
5,ATL,118.5,82
6,MIN,118.0,82
7,DET,117.8,82
8,UTA,117.6,82
9,NYK,116.5,82


In [ ]:
# 2: Which team has the best home record?

# CASE WHEN: if else statement
# SUM: adds 1s and 0s
# AVG: average of 1/0 values for win percentage
duckdb.sql("""
SELECT home_team AS team,
       SUM(CASE WHEN home_pts > away_pts THEN 1 ELSE 0 END) AS home_wins,
       SUM(CASE WHEN home_pts < away_pts THEN 1 ELSE 0 END) AS home_losses,
       ROUND(AVG(CASE WHEN home_pts > away_pts THEN 1.0 ELSE 0.0 END), 3) AS home_win_pct
FROM games
WHERE NOT neutral_site
GROUP BY home_team
ORDER BY home_win_pct DESC
""").df()

,team,home_wins,home_losses,home_win_pct
0,OKC,34.0,7.0,0.829
1,SAS,32.0,8.0,0.800
2,DET,31.0,9.0,0.775
3,NYK,30.0,10.0,0.750
4,HOU,30.0,11.0,0.732
5,BOS,30.0,11.0,0.732
6,LAL,28.0,13.0,0.683
7,DEN,28.0,13.0,0.683
8,CLE,27.0,14.0,0.659
9,MIA,26.0,15.0,0.634


In [ ]:
# 3: What’s the home-court edge?

duckdb.sql("""
SELECT ROUND(AVG(home_pts - away_pts), 2) AS home_edge,
       COUNT(*) AS games
FROM games
WHERE NOT neutral_site
""").df()

,home_edge,games
0,1.73,1225


In [ ]:
# 4: Each team’s average margin over its last 10 games

# WITH ranked AS(): temporary table named rank for this query
# ROW NUMBER() OVER (PARTITION BY team ORDER BY GAME_DATE DESC): rank each row by team and order by game date descending
duckdb.sql("""
WITH ranked AS (
    SELECT team,
           GAME_DATE,
           pts - opp_pts AS margin,
           ROW_NUMBER() OVER (PARTITION BY team ORDER BY GAME_DATE DESC) AS games_ago
    FROM team_games
)
SELECT team,
       ROUND(AVG(margin), 1) AS last10_margin
FROM ranked
WHERE games_ago <= 10
GROUP BY team
ORDER BY last10_margin DESC
""").df()

,team,last10_margin
0,HOU,15.2
1,SAS,13.6
2,POR,13.1
3,TOR,12.2
4,DET,11.8
5,BOS,11.2
6,DEN,10.5
7,OKC,10.3
8,ATL,6.4
9,CHA,6.3


In [ ]:
# 5: Longest winning streak for each team

# SELECT team, GAME_DATE, pts > opp_pts AS won: create a column that is true if the team won
# WHERE WON: filter for only the games that were won
duckdb.sql("""
WITH results AS (
    SELECT team,
           GAME_DATE,
           pts > opp_pts AS won,
           ROW_NUMBER() OVER (PARTITION BY team ORDER BY GAME_DATE) AS game_num
    FROM team_games
),
wins AS (
    SELECT team,
           GAME_DATE,
           game_num - ROW_NUMBER() OVER (PARTITION BY team ORDER BY GAME_DATE) AS streak_id
    FROM results
    WHERE won
)
SELECT team,
       COUNT(*)       AS streak_length,
       MIN(GAME_DATE) AS started,
       MAX(GAME_DATE) AS ended
FROM wins
GROUP BY team, streak_id
ORDER BY streak_length DESC
LIMIT 5
""").df()